## Built-in callbacks

1. TensorBoard
2. Model checkpoints
3. Early Stopping
4. CSV Logger
5. Learning Rate Scheduler
6. ReduceLROnPlateau

In [1]:
import tensorflow as tf
import numpy as np
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import io
from PIL import Image

from tensorflow.keras.callbacks import TensorBoard, ModelCheckpoint, EarlyStopping, CSVLogger, LearningRateScheduler, ReduceLROnPlateau
%load_ext tensorboard

import os
import math
import datetime
import pandas as pd

### Prepare the Horses vs Humans dataset

In [2]:
path = "./data"
splits, info = tfds.load('horses_or_humans',
                         data_dir=path,
                         as_supervised=True,
                         with_info=True, split=['train[:80%]', 'train[80%:]', 'test'])

(train_examples, validation_examples, test_examples) = splits
num_examples = info.splits['train'].num_examples
num_classes = info.features['label'].num_classes

2026-02-17 15:16:01.038524: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2026-02-17 15:16:01.038695: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-02-17 15:16:01.038701: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
I0000 00:00:1771321561.039011 1034382 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1771321561.039283 1034382 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [3]:
SIZE = 150
IMAGE_SIZE = (SIZE, SIZE)

In [4]:
# Format the images to feed into model

def format_image(image, label):
    image = tf.image.resize(image, IMAGE_SIZE) / 255.0
    return image, label

BATCH_SIZE = 32

In [5]:
train_batches = train_examples.shuffle(num_examples//4).map(format_image).batch(BATCH_SIZE).prefetch(1)
validation_batches = validation_examples.map(format_image).batch(BATCH_SIZE).prefetch(1)
test_batches = test_examples.map(format_image).batch(1)

In [6]:
for image_batch, label_batch in train_batches.take(1):
    pass

print("Image batch shape: ",image_batch.shape)
print("Label batch shape: ",label_batch.shape)

2026-02-17 15:16:01.334326: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:387] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608


Image batch shape:  (32, 150, 150, 3)
Label batch shape:  (32,)


2026-02-17 15:16:01.546257: W tensorflow/core/kernels/data/cache_dataset_ops.cc:916] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.
2026-02-17 15:16:01.550851: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### Build model

In [7]:
def build_model(dense_units, input_shape=IMAGE_SIZE+(3,)):
    model = tf.keras.models.Sequential([
        tf.keras.layers.Conv2D(16, (3,3), activation='relu', input_shape=input_shape),
        tf.keras.layers.MaxPooling2D(2,2),
        tf.keras.layers.Conv2D(32, (3,3), activation='relu'),
        tf.keras.layers.MaxPooling2D(2,2),
        tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
        tf.keras.layers.MaxPooling2D(2,2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(dense_units, activation='relu'),
        tf.keras.layers.Dense(2, activation='softmax')
    ])
    return model

## TensorBoard

In [8]:
!rm -rf logs

In [14]:
model = build_model(dense_units=256)
model.compile(optimizer='sgd',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

logdir = os.path.join("logs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
tensorboard_callback = tf.keras.callbacks.TensorBoard(logdir)

model.fit(train_batches, epochs=10, validation_data=validation_batches, callbacks=[tensorboard_callback])

Epoch 1/10


/Users/opmule/miniforge3/envs/ai_tensorflow/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 55ms/step - accuracy: 0.5132 - loss: 0.6924 - val_accuracy: 0.4390 - val_loss: 0.6927
Epoch 2/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.6209 - loss: 0.6356 - val_accuracy: 0.7171 - val_loss: 0.6091
Epoch 3/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.6862 - loss: 0.5924 - val_accuracy: 0.7073 - val_loss: 0.5706
Epoch 4/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.7385 - loss: 0.5488 - val_accuracy: 0.8098 - val_loss: 0.4589
Epoch 5/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.7845 - loss: 0.4662 - val_accuracy: 0.8341 - val_loss: 0.4364
Epoch 6/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.8117 - loss: 0.4336 - val_accuracy: 0.8488 - val_loss: 0.3714
Epoch 7/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.8497 - loss: 0.3589 - val_accuracy: 0.8780 - val_loss: 0.3084
Epoch 8/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.8552 - loss: 0.3333 - val_accuracy: 0.8878 - val_loss: 0.

In [15]:
%tensorboard --logdir logs

Reusing TensorBoard on port 6006 (pid 45920), started 0:00:58 ago. (Use '!kill 45920' to kill it.)